In [1]:
%load_ext autoreload
%autoreload 2
%env CUPY_ACCELERATORS=cutensor,cub

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: CUPY_ACCELERATORS=cutensor,cub


In [2]:
import tensorly as tl
import plotly.io as pio
#pio.renderers.default = 'iframe'
tl.set_backend('cupy')
tl.tenalg.set_backend('einsum')
tl.plugins.use_opt_einsum()
print(f'TensorLy backend: {tl.get_backend()}')

TensorLy backend: cupy


In [3]:
from moabb.datasets import *
from moabb.paradigms import P300

dataset = BNCI2014_008()
paradigm = P300()
X, y, meta = paradigm.get_data(dataset)
X = tl.tensor(X)
X.shape

/usr/local/share/venv/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 4200 events (all good), 0 – 1 s (baseline off), ~65.9 MiB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")
/usr/local/share/venv/lib64/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/usr/local/share/venv/lib/python3.12/site-packages/moabb/datasets/preprocessing.py:279: UserWarning: warnEpochs <Epochs | 4200 events (all good), 0 – 1 s (baseline off), ~65.9 MiB, data loaded,
 'Target': 700
 'NonTarget': 3500>
  warn(f"warnEpochs {epochs}")
/usr/local/share/venv/lib64/python3.12/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate argum

(33600, 8, 257)

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from hoda.classification import SelectFCutoff
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

clf = make_pipeline(
    FunctionTransformer(tl.to_numpy),
    PCA(n_components=None, whiten=True),
    SelectFCutoff(cutoff=1),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr')
)

In [5]:
hoda_params=  dict(
    max_iter=64,
    toeplitz=(1,),
    taper=False,
    verbose=True,
    refit_shrinkage=True,
)

In [6]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5)

bttdacv_params = dict(
    hoda_params=hoda_params,
    verbose=True,
    cv=cv,
    n_jobs=1,
    clf = clf,
)

In [7]:
from hoda.classification import BTTDACV

bttdacv = BTTDACV(
    max_n_blocks=2,
    fixed_n_blocks=True,
    thetas=[0,0.1,0.2,0.3,0.4,0.5,0.6, 0.7, 0.8, 0.9,1],
    **bttdacv_params
)

In [ ]:
bttdacv.fit(X,y)

fold=0, theta=0
Fitting block 1/2...


Forward model :   4%|█▋                                        | 5/127 [00:00<00:10, 11.24it/s]


Fitting block 2/2...


Forward model :   3%|█▎                                        | 4/127 [00:00<00:12, 10.14it/s]


fold=0, theta=0, n_blocks=2
fold=0, theta=0.1
Fitting block 1/2...


...

fold=4, theta=0.9, n_blocks=2
fold=4, theta=1
Fitting block 1/2...


Forward model : 100%|████████████████████████████████████████| 127/127 [02:26<00:00,  1.16s/it]


Fitting block 2/2...


Forward model :  11%|████▌                                    | 14/127 [00:17<02:24,  1.27s/it]


fold=4, theta=1, n_blocks=2
> /home/arne/Workspace/PhD/hoda-bci/src/hoda/classification.py(171)fit()
    169         pdb.set_trace()
    170 
--> 171         if self.fixed_n_blocks:
    172             opt_n_blocks = self.max_n_blocks
    173 

